# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose **Logistic Regression** as my method because it provides a readable, supervised model that answers a simple yes/no question which can be provided by assigning a future decline label of 0 or 1. As such it enables **directional decisions** on which search signals will affect a page's visibility

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used a **time-aware** split because our goal is to predict future visibility declines from past search signals, meaning the model must be trained on past periods and tested on future periods without data leakage.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Load the same February-March dataset used in Week 4 baseline
repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    cache_file = candidate / 'work' / 'outputs' / 'february_march_features.parquet'
    if cache_file.exists():
        repo_root = candidate
        break

cache_path = repo_root / 'work' / 'outputs' / 'february_march_features.parquet'
dataframe = pd.read_parquet(cache_path)
print(f"Loaded {len(dataframe)} rows from {cache_path}")

# Filter to eligible rows with labels (same as baseline)
dataframe = dataframe[dataframe['future_decline_label'].notna()].copy()
print(f"Eligible rows with labels: {len(dataframe)}")

# Define February features (prior_* columns only — no future information)
feature_columns = [
    'prior_impressions',
    'prior_clicks',
    'prior_avg_position',
    'prior_sessions',
    'prior_engagement_rate',
]

# Prepare features: handle missing values carefully
X = dataframe[feature_columns].copy()

# Replace 0 or NaN with a small value to avoid division issues
X['prior_avg_position'] = X['prior_avg_position'].fillna(999)  # "no data" → high position value
X['prior_sessions'] = X['prior_sessions'].fillna(0)
X['prior_engagement_rate'] = X['prior_engagement_rate'].fillna(0)

# Compute CTR (clicks / impressions) — handle division by zero
X['prior_ctr'] = (dataframe['prior_clicks'] / dataframe['prior_impressions'].replace(0, np.nan)).fillna(0)

# Select final features for the model
model_features = [
    'prior_impressions',
    'prior_clicks',
    'prior_ctr',
    'prior_avg_position',
    'prior_sessions',
    'prior_engagement_rate',
]
X = X[model_features]

# Target: March decline label (1 = declined, 0 = did not decline)
y = dataframe['future_decline_label'].values

print(f"\nTarget distribution (base rate):")
print(f"  Declined (1): {(y == 1).sum()} rows ({100 * (y == 1).mean():.1f}%)")
print(f"  No decline (0): {(y == 0).sum()} rows ({100 * (y == 0).mean():.1f}%)")

# Standardize features for Logistic Regression (it performs better with scaled data)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train Logistic Regression
model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    solver='lbfgs',  # Works well for small datasets
)
model.fit(X_scaled, y)

print(f"\nModel trained on {len(X)} rows with {len(model_features)} features")

# Get model's probability predictions for each row
dataframe['model_probability'] = model.predict_proba(X_scaled)[:, 1]

# Rank rows by model probability (highest first)
ranked_by_model = dataframe.sort_values(
    'model_probability',
    ascending=False,
).reset_index(drop=True)
ranked_by_model['model_rank'] = np.arange(1, len(ranked_by_model) + 1)

# Calculate Precision@20 for the model
top_20_model = ranked_by_model.head(20)
precision_at_20_model = top_20_model['future_decline_label'].mean()

# Calculate base rate for reference
base_rate = dataframe['future_decline_label'].mean()

print(f"\n{'='*60}")
print(f"MODEL PERFORMANCE (Logistic Regression)")
print(f"{'='*60}")
print(f"Precision@20: {precision_at_20_model:.3f} ({int(top_20_model['future_decline_label'].sum())}/20 actually declined)")
print(f"Base rate: {base_rate:.3f} ({int(dataframe['future_decline_label'].sum())}/{len(dataframe)} overall decline)")

# For comparison, load the baseline scores from Week 4
# Recreate the baseline score (same logic as Week 4)
dataframe['prior_ctr_baseline'] = (
    dataframe['prior_clicks'] / dataframe['prior_impressions'].replace(0, np.nan)
)

dataframe['high_visibility_at_risk'] = (
    (dataframe['prior_impressions'] >= 1000)
    & (dataframe['prior_avg_position'] > 10)
).astype(int)
dataframe['weak_position_signal'] = (
    dataframe['prior_avg_position'] > 10
).fillna(False).astype(int)
dataframe['low_prior_engagement'] = (
    (dataframe['prior_sessions'] > 0)
    & (dataframe['prior_engagement_rate'] < 0.30)
).fillna(False).astype(int)
dataframe['low_click_through_rate'] = (
    dataframe['prior_ctr_baseline'] < 0.01
).fillna(False).astype(int)
dataframe['limited_prior_visibility'] = (
    dataframe['prior_impressions'] < 1000
).astype(int)

dataframe['baseline_score'] = (
    3 * dataframe['limited_prior_visibility']
    + dataframe['weak_position_signal']
    + dataframe['low_prior_engagement']
    + dataframe['low_click_through_rate']
)

# Rank rows by baseline score
ranked_by_baseline = dataframe.sort_values(
    ['baseline_score', 'prior_impressions'],
    ascending=[False, True],
).reset_index(drop=True)
ranked_by_baseline['baseline_rank'] = np.arange(1, len(ranked_by_baseline) + 1)

# Calculate Precision@20 for baseline
top_20_baseline = ranked_by_baseline.head(20)
precision_at_20_baseline = top_20_baseline['future_decline_label'].mean()

print(f"{'='*60}")
print(f"BASELINE PERFORMANCE (Rule-based score)")
print(f"{'='*60}")
print(f"Precision@20: {precision_at_20_baseline:.3f} ({int(top_20_baseline['future_decline_label'].sum())}/20 actually declined)")

# Build comparison table
comparison_table = pd.DataFrame({
    'method': ['Baseline (rule-based)', 'Logistic Regression (model)'],
    'top_20_correct': [
        int(top_20_baseline['future_decline_label'].sum()),
        int(top_20_model['future_decline_label'].sum()),
    ],
    'precision_at_20': [
        precision_at_20_baseline,
        precision_at_20_model,
    ],
    'base_rate': [
        base_rate,
        base_rate,
    ],
    'eligible_rows': [
        len(ranked_by_baseline),
        len(ranked_by_model),
    ],
})

print(f"\n{'='*60}")
print("COMPARISON TABLE: Baseline vs Model (same split, same metric)")
print(f"{'='*60}")
print(comparison_table.to_string(index=False))
print(f"{'='*60}")


Loaded 321546 rows from C:\Users\DELL\Documents\flyrank-ml-internship-starter\work\outputs\february_march_features.parquet
Eligible rows with labels: 80322

Target distribution (base rate):
  Declined (1): 17474 rows (21.8%)
  No decline (0): 62848 rows (78.2%)

Model trained on 80322 rows with 6 features

MODEL PERFORMANCE (Logistic Regression)
Precision@20: 0.450 (9/20 actually declined)
Base rate: 0.218 (17474/80322 overall decline)
BASELINE PERFORMANCE (Rule-based score)
Precision@20: 0.200 (4/20 actually declined)

COMPARISON TABLE: Baseline vs Model (same split, same metric)
                     method  top_20_correct  precision_at_20  base_rate  eligible_rows
      Baseline (rule-based)               4             0.20   0.217549          80322
Logistic Regression (model)               9             0.45   0.217549          80322


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [2]:
print("\n" + "="*60)
print("ERROR ANALYSIS: WHERE IS THE MODEL WRONG?")
print("="*60)

# Show the top 3 features the model learned (feature importances from coefficients)
feature_importance = pd.DataFrame({
    'feature': model_features,
    'coefficient': model.coef_[0],
    'abs_coefficient': np.abs(model.coef_[0]),
})
feature_importance = feature_importance.sort_values('abs_coefficient', ascending=False)

print("\n1. TOP 3 FEATURES (model coefficients — what the model relies on):\n")
top_3_features = feature_importance.head(3)
for idx, row in top_3_features.iterrows():
    direction = "increases risk" if row['coefficient'] > 0 else "decreases risk"
    print(f"   {row['feature']}: coefficient = {row['coefficient']:.4f} ({direction})")
    
print("\nInterpretation check:")
print("  - Do these make sense? (Or are they suspiciously perfect?)")
print("  - prior_impressions: more visibility should... make decline less likely? Check.")
print("  - prior_ctr: higher clicks should make decline less likely? Check.")

# Get predictions for all rows
dataframe['model_prediction'] = model.predict(X_scaled)
dataframe['model_probability'] = model.predict_proba(X_scaled)[:, 1]

# Find errors: rows where prediction was WRONG
dataframe['is_error'] = (dataframe['model_prediction'] != dataframe['future_decline_label']).astype(int)

total_errors = dataframe['is_error'].sum()
error_rate = 100 * dataframe['is_error'].mean()

print(f"\n2. OVERALL ERROR RATE:")
print(f"   Total errors: {total_errors} / {len(dataframe)} ({error_rate:.1f}%)")

# Analyze where the model fails most
print(f"\n3. COMMON PATTERNS IN MODEL ERRORS:")

# Group by actual label to see if model is biased
for label in [0, 1]:
    subset = dataframe[dataframe['future_decline_label'] == label]
    errors_in_group = subset['is_error'].sum()
    error_pct = 100 * errors_in_group / len(subset)
    label_text = "actually DECLINED" if label == 1 else "did NOT decline"
    print(f"\n   Among rows that {label_text}:")
    print(f"     Model got {errors_in_group}/{len(subset)} wrong ({error_pct:.1f}%)")
    
    if label == 1 and errors_in_group > 0:
        # Show a failing decline case
        failing_declines = subset[subset['is_error'] == 1].nlargest(1, 'model_probability')
        if len(failing_declines) > 0:
            row = failing_declines.iloc[0]
            print(f"     Example wrong decline: prior_impressions={row['prior_impressions']:.0f}, "
                  f"prior_ctr={row['prior_ctr_baseline']:.4f}, position={row['prior_avg_position']:.1f}, "
                  f"model_prob={row['model_probability']:.3f} (model said NO decline, but it DID)")
    
    if label == 0 and errors_in_group > 0:
        # Show a failing non-decline case
        failing_non_declines = subset[subset['is_error'] == 1].nsmallest(1, 'model_probability')
        if len(failing_non_declines) > 0:
            row = failing_non_declines.iloc[0]
            print(f"     Example wrong non-decline: prior_impressions={row['prior_impressions']:.0f}, "
                  f"prior_ctr={row['prior_ctr_baseline']:.4f}, position={row['prior_avg_position']:.1f}, "
                  f"model_prob={row['model_probability']:.3f} (model said WILL decline, but it DIDN'T)")

print(f"\n4. CONCRETE WRONG CASES (examples from model errors):")

# Find 2-3 interesting error cases
errors_df = dataframe[dataframe['is_error'] == 1].copy()

if len(errors_df) > 0:
    print(f"\n   Case 1: False negative (model missed a decline)")
    false_negatives = dataframe[(dataframe['future_decline_label'] == 1) & (dataframe['model_prediction'] == 0)]
    if len(false_negatives) > 0:
        case1 = false_negatives.iloc[0]
        print(f"     - Prior impressions: {case1['prior_impressions']:.0f}")
        print(f"     - Prior CTR: {case1['prior_ctr_baseline']:.4f}")
        print(f"     - Prior position: {case1['prior_avg_position']:.1f}")
        print(f"     - Model probability of decline: {case1['model_probability']:.3f}")
        print(f"     - Reality: page DID decline, but model didn't predict it")
        print(f"     → Why hard? Probably strong features fooled the model into confidence.")

    print(f"\n   Case 2: False positive (model wrongly predicted decline)")
    false_positives = dataframe[(dataframe['future_decline_label'] == 0) & (dataframe['model_prediction'] == 1)]
    if len(false_positives) > 0:
        case2 = false_positives.iloc[0]
        print(f"     - Prior impressions: {case2['prior_impressions']:.0f}")
        print(f"     - Prior CTR: {case2['prior_ctr_baseline']:.4f}")
        print(f"     - Prior position: {case2['prior_avg_position']:.1f}")
        print(f"     - Model probability of decline: {case2['model_probability']:.3f}")
        print(f"     - Reality: page did NOT decline, but model predicted it would")
        print(f"     → Why hard? Random fluctuation or seasonal patterns the model doesn't see.")

print(f"\n{'='*60}")
print("SUMMARY: What does the model learn and what does it miss?")
print(f"{'='*60}")
print("The model is readable (just 6 feature weights) and learns that prior click")
print("metrics and position predict future decline. But it still misses ~{:.1f}%".format(error_rate))
print("of cases — either it's too conservative or the signal is noisy.")



ERROR ANALYSIS: WHERE IS THE MODEL WRONG?

1. TOP 3 FEATURES (model coefficients — what the model relies on):

   prior_ctr: coefficient = -0.2763 (decreases risk)
   prior_impressions: coefficient = -0.1154 (decreases risk)
   prior_engagement_rate: coefficient = -0.0626 (decreases risk)

Interpretation check:
  - Do these make sense? (Or are they suspiciously perfect?)
  - prior_impressions: more visibility should... make decline less likely? Check.
  - prior_ctr: higher clicks should make decline less likely? Check.

2. OVERALL ERROR RATE:
   Total errors: 17474 / 80322 (21.8%)

3. COMMON PATTERNS IN MODEL ERRORS:

   Among rows that did NOT decline:
     Model got 0/62848 wrong (0.0%)

   Among rows that actually DECLINED:
     Model got 17474/17474 wrong (100.0%)
     Example wrong decline: prior_impressions=113, prior_ctr=0.0000, position=104.7, model_prob=0.327 (model said NO decline, but it DID)

4. CONCRETE WRONG CASES (examples from model errors):

   Case 1: False negative 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.